Hoe werkt het model? 
- Data verdelen in 3 groepen (zoals bij KNN)
- Standaardiseren features
- Bereken voor elke klasse een gemiddelde featurevector (soft labels als gewichten)
- Bereken de gezamenlijke spreiding binnen klassen
- Voor elk EEG de waarschijnlijkheid van de zes klassen voorspellen
- Bereken KL-score

Verschillende waarden testen voor "shrinkage" (=vorm van regularisatie die de covariantiematrix stabieler kan maken)

In [1]:
# controle van mijn bestanden 

from pathlib import Path
import os

print("Beschikbare datasets:")
print(os.listdir("/kaggle/input"))

print("\nZoeken naar HMS-train.csv:")
for file in Path("/kaggle/input").rglob("train.csv"):
    print(file)

print("\nZoeken naar het baseline-script:")
for file in Path("/kaggle/input").rglob("hms_sklearn_baseline.py"):
    print(file)

Beschikbare datasets:
['datasets', 'competitions']

Zoeken naar HMS-train.csv:
/kaggle/input/competitions/hms-harmful-brain-activity-classification/train.csv

Zoeken naar het baseline-script:
/kaggle/input/datasets/hannaleysen/link-github-exp-4/src/hms_sklearn_baseline.py


In [2]:
#bestaande baseline hergebruiken ! enkel de dingen veranderen die je meot veranderen
# Eerst de functies uit de baseline eventjes controleren

from pathlib import Path
import importlib.util
import inspect

# Python-Bestand zoeken in GitHub-dataset
scripts = list(
    Path("/kaggle/input").rglob("hms_sklearn_baseline.py")
)

if len(scripts) != 1:
    raise FileNotFoundError(
        f"Verwacht één baselinebestand, gevonden: {scripts}"
    )

SCRIPT_PATH = scripts[0]
print("Baseline gevonden op:", SCRIPT_PATH)
# Importeer bestaande baseline als Python-module
spec = importlib.util.spec_from_file_location(
    "hms_baseline",
    SCRIPT_PATH
)

baseline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baseline)

print("Baseline succesvol geïmporteerd!")
print("\nBeschikbare functies:")

for name in [
    "load_labels",
    "build_features",
    "kl_score",
    "cross_validate",
    "make_model",
    "to_matrix",
    "expand"
]:
    function = getattr(baseline, name, None)
    if function is not None:
        print(name, inspect.signature(function))


Baseline gevonden op: /kaggle/input/datasets/hannaleysen/link-github-exp-4/src/hms_sklearn_baseline.py
Baseline succesvol geïmporteerd!

Beschikbare functies:
load_labels (data_dir, max_specs=None, seed=42)
build_features (meta, spec_dir, is_test=False, n_jobs=-1)
kl_score (y_true, y_pred, eps=1e-15)
cross_validate (X, Y, groups, num_votes, n_folds, seed)
make_model (name, seed=42)
to_matrix (frame, feats)
expand (X, Y)


In [3]:
# KNN --> de 6 expertkansen rechtstreeks gebruiken en niet eerst omzetten naar gewone klassevoorbeelden 

import inspect

print("Informatie over build_features:")
print(inspect.getsource(baseline.build_features))

print("\nInformatie over cross_validate:")
print(inspect.getsource(baseline.cross_validate))


Informatie over build_features:
def build_features(meta, spec_dir, is_test=False, n_jobs=-1):
    """Compute features for all EEGs in `meta`, reading each spectrogram file once (in parallel)."""
    t0 = time.time()
    if is_test:   # test.csv has no label offsets
        meta = meta.assign(off_min=0.0, off_max=0.0)
    jobs = [delayed(spectrogram_features)(spec_dir / f"{sid}.parquet",
                                          list(zip(g.eeg_id, g.off_min, g.off_max)), is_test)
            for sid, g in meta.groupby("spectrogram_id")]
    results = Parallel(n_jobs=n_jobs, batch_size=32)(jobs)
    out = pd.DataFrame([row for rows in results for row in rows])
    print(f"{len(jobs)} spectrograms -> {len(out)} feature rows in {time.time() - t0:.0f}s")
    return out


Informatie over cross_validate:
def cross_validate(X, Y, groups, num_votes, n_folds, seed):
    """Patient-grouped K-fold CV for both models; prints scores and returns the better model's name."""
    prior = np.tile(Y.mean(

In [4]:
# KNN-experiment 


In [5]:
# eerst nog dingetjes ophalen uit de baseline 
import inspect

print("MODEL_NAMES:")
print(baseline.MODEL_NAMES)              # welke modellen vergelijkt de baseline momenteel

print("\nFunctie fit:")
print(inspect.getsource(baseline.fit))   # Hoe worden bestaande modellen getraind 

print("\nFunctie load_labels:")
print(inspect.getsource(baseline.load_labels))  # hoe worden de 6 doelklassen ingelezen

print("\nFunctie to_matrix:")
print(inspect.getsource(baseline.to_matrix))    # om exact dezelfde featureselectie als de baseline te gebruiken

MODEL_NAMES:
['logreg', 'hgb']

Functie fit:
def fit(name, X, Y, seed=42):
    """Fit model `name` on features X and soft labels Y (rows are vote distributions)."""
    Xe, ye, we = expand(X, Y)
    m = make_model(name, seed)
    if name == "logreg":   # in a Pipeline the weight must be routed to the final step
        m.fit(Xe, ye, logisticregression__sample_weight=we)
    else:
        m.fit(Xe, ye, sample_weight=we)
    return m


Functie load_labels:
def load_labels(data_dir, max_specs=None, seed=42):
    """Return one row per EEG with its spectrogram id, label offsets, patient id and vote distribution."""
    train = pd.read_csv(data_dir / "train.csv")

    # train.csv has several labelled sub-segments per eeg_id: sum their votes and keep the offset range.
    agg = {"spectrogram_id": "first", "spectrogram_label_offset_seconds": ["min", "max"], "patient_id": "first"}
    agg.update({c: "sum" for c in CLASSES})
    df = train.groupby("eeg_id").agg(agg)
    df.columns = ["spectrogra

In [6]:
# Hoe brengt de baseline alles samen?  --> ervoor zorgen dat KNN dezelfde invoergegevens krijgt als de andere 2 modellen in de baseline 
import inspect

print("1. De functie main:")  
print(inspect.getsource(baseline.main))   # volgorde dataverwerking baseline (gehele workflow)

print("\n2. De functie make_model:")
print(inspect.getsource(baseline.make_model))   # hoe modellen instellen 

print("\n3. De beschikbare klassen:")
print(baseline.CLASSES)                   # volgorde doelklassen bevestigen (KNN moet deze ook hebben)

1. De functie main:
def main():
    parser = argparse.ArgumentParser(description="HMS simple scikit-learn baseline")
    parser.add_argument("--data-dir", type=Path, required=True,
                        help="folder that contains train.csv, test.csv, train_spectrograms/ and test_spectrograms/")
    parser.add_argument("--mode", choices=["full", "quick"], default="full",
                        help="full: all training spectrograms; quick: a random 3,000 for a fast check")
    parser.add_argument("--no-cv", action="store_true", help="skip cross-validation and use gradient boosting")
    parser.add_argument("--folds", type=int, default=5, help="number of cross-validation folds")
    parser.add_argument("--n-jobs", type=int, default=-1, help="CPU cores for reading spectrograms (-1 = all)")
    parser.add_argument("--output", type=Path, default=Path("submission.csv"), help="where to write predictions")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

Schalen is voor KNN zeer belangrijk --> berekent de afstanden tussen datapunten. Zonder schaling kunnen kenmerken met grote waarden de afstand beïnvloeden op een slechte manier 

Volgorde behouden: ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote'] 

# KNN Code

In [7]:
# data inladen en features bouwen (zelfde gebruiken als baseline)
from pathlib import Path
import numpy as np
import pandas as pd

# Pad naar de HMS-competitiedataset
DATA_DIR = Path(
    "/kaggle/input/competitions/hms-harmful-brain-activity-classification"
)

# Controleer of de dataset gevonden wordt
assert (DATA_DIR / "train.csv").exists(), "train.csv niet gevonden!"

# 1. Labels inladen
df = baseline.load_labels(
    DATA_DIR,
    max_specs=3000,
    seed=42
)

print(df.shape, "EEGs")
print(df.patient_id.nunique(), "verschillende patiënten")
print(df.spectrogram_id.nunique(), "spectrogrammen")

# 2. Features berekenen
X_df = baseline.build_features(
    df,
    DATA_DIR / "train_spectrograms",
    n_jobs=2
)

# 3. Labels en features samenvoegen
df = df.merge(X_df, on="eeg_id")

# 4. Dezelfde featureselectie gebruiken als de baseline
feats = [c for c in X_df.columns if c != "eeg_id"]

X = baseline.to_matrix(df, feats)
Y = df[baseline.CLASSES].to_numpy(dtype=np.float64)

groups = df["patient_id"].to_numpy()
num_votes = df["num_votes"].to_numpy()

print("\nData klaar voor KNN!")
print("Aantal EEG-records:", len(X))
print("Aantal features:", X.shape[1])
print("Aantal klassen:", Y.shape[1])
print("Vorm van X:", X.shape)
print("Vorm van Y:", Y.shape)

(4738, 12) EEGs
1163 verschillende patiënten
3000 spectrogrammen
3000 spectrograms -> 4738 feature rows in 87s

Data klaar voor KNN!
Aantal EEG-records: 4738
Aantal features: 88
Aantal klassen: 6
Vorm van X: (4738, 88)
Vorm van Y: (4738, 6)


In [8]:
# LDA classifier bouwen 


import numpy as np

class SoftLDAClassifier:

    def __init__(self, shrinkage=0.1, ridge=1e-4):
        self.shrinkage = shrinkage
        self.ridge = ridge
        self.means = None
        self.priors = None
        self.coef = None
        self.intercept = None

    def fit(self, X, Y):
        """
        X = featurematrix
        Y = soft labels met zes klasseverdelingen
        """

        X = np.asarray(X, dtype=np.float64)
        Y = np.asarray(Y, dtype=np.float64)

        n_samples, n_features = X.shape
        n_classes = Y.shape[1]

        # 1. Gemiddelde per klasse berekenen
        class_mass = Y.sum(axis=0)
        self.priors = class_mass / class_mass.sum()
        self.means = np.zeros((n_classes, n_features))

        for c in range(n_classes):

            weights = Y[:, c]
            self.means[c] = (
                np.sum(X * weights[:, None], axis=0)
                / max(class_mass[c], 1e-12)
            )
            
        # 2. Covariantiematrix berekenen
        covariance = np.zeros((n_features, n_features))

        for c in range(n_classes):

            # Afstand van iedere observatie tot het gemiddelde van de huidige klasse.
            differences = X - self.means[c]

            # Soft labels bepalen hoeveel iedere observatie meetelt voor deze klasse.
            weighted_differences = differences * Y[:, c, None]
            covariance += weighted_differences.T @ differences

        covariance /= max(Y.sum(), 1e-12)

        # 3. Shrinkage toepassen
        diagonal = np.diag(np.diag(covariance))
        covariance = (
            (1 - self.shrinkage) * covariance
            + self.shrinkage * diagonal
        )

        # Extra regularisatie 
        covariance += self.ridge * np.eye(n_features)

        # 4. Classificatiecoëfficiënten berekenen
              # lineaire stelsel oplossen ipv inverse matrix expliciet berekenen
        self.coef = np.linalg.solve(
            covariance,
            self.means.T).T

        self.intercept = (
            -0.5 * np.sum(self.means * self.coef, axis=1)
            + np.log(np.maximum(self.priors, 1e-12)))

        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float64)

        # Scores voor de zes klassen.
        scores = X @ self.coef.T + self.intercept

        # Softmax: scores omzetten naar kansen.
        scores -= np.max(scores, axis=1, keepdims=True)
        probabilities = np.exp(scores)
        probabilities /= probabilities.sum(
            axis=1,
            keepdims=True
        )

        return probabilities


In [9]:
# Cross-validatie en KL-scores 
    # 4 shrinkage-instellingen testen --> 4 verschillende LDA modellen trainen en vergelijken 

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

# Instellingen
N_FOLDS = 3
SHRINKAGE_VALUES = [0.0, 0.1, 0.3, 0.5]

gkf = GroupKFold(n_splits=N_FOLDS)

# Bewaren van out-of-fold voorspellingen.
lda_oof = {}

for shrinkage in SHRINKAGE_VALUES:
    lda_oof[shrinkage] = np.zeros_like(Y, dtype=np.float64)


# Cross-validation
for fold, (tr, va) in enumerate(
    gkf.split(X, Y, groups),
    start=1):

    print(f"\n========== FOLD {fold} ==========")

    # 1. Features standaardiseren
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[tr])
    X_valid = scaler.transform(X[va])

    # 2. Alle shrinkage-instellingen testen.
    for shrinkage in SHRINKAGE_VALUES:
        print(f"Training LDA met shrinkage={shrinkage}")

        # Model aanmaken.
        model = SoftLDAClassifier(
            shrinkage=shrinkage,
            ridge=1e-4)

        # Trainen met de soft labels.
        model.fit(X_train, Y[tr])

        # Voorspellingen op de validatieset + bewaren 
        predictions = model.predict_proba(X_valid)
        lda_oof[shrinkage][va] = predictions

        # KL-score berekenen.
        fold_score = baseline.kl_score(
            Y[va],
            predictions)

        print(
            f"Fold {fold} | "
            f"Shrinkage {shrinkage} | "
            f"KL-score: {fold_score:.5f}")

    print(f"Fold {fold} afgerond.")


# Totale resultaten 

results = []

for shrinkage in SHRINKAGE_VALUES:
    predictions = lda_oof[shrinkage]
    score = baseline.kl_score(Y, predictions)
    results.append({
        "Experiment": f"LDA (shrinkage={shrinkage})",
        "Model": "Soft LDA / FDA",
        "KL_score": score})

# Resultaten DataFrame
results_df = pd.DataFrame(results)

# Beste resultaat bovenaan
results_df = results_df.sort_values(
    "KL_score",
    ascending=True
).reset_index(drop=True)

# Resultaten tonen

print("\n========== LDA / FDA RESULTATEN ==========")

print(results_df.to_string(index=False))

best_result = results_df.iloc[0]

print("\nBeste LDA-instelling:")
print(best_result.to_string())




========== FOLD 1 ==========
Training LDA met shrinkage=0.0
Fold 1 | Shrinkage 0.0 | KL-score: 1.21830
Training LDA met shrinkage=0.1
Fold 1 | Shrinkage 0.1 | KL-score: 1.14842
Training LDA met shrinkage=0.3
Fold 1 | Shrinkage 0.3 | KL-score: 1.16277
Training LDA met shrinkage=0.5
Fold 1 | Shrinkage 0.5 | KL-score: 1.20534
Fold 1 afgerond.

========== FOLD 2 ==========
Training LDA met shrinkage=0.0
Fold 2 | Shrinkage 0.0 | KL-score: 1.26919
Training LDA met shrinkage=0.1
Fold 2 | Shrinkage 0.1 | KL-score: 1.22616
Training LDA met shrinkage=0.3
Fold 2 | Shrinkage 0.3 | KL-score: 1.23315
Training LDA met shrinkage=0.5
Fold 2 | Shrinkage 0.5 | KL-score: 1.27066
Fold 2 afgerond.

========== FOLD 3 ==========
Training LDA met shrinkage=0.0
Fold 3 | Shrinkage 0.0 | KL-score: 1.24932
Training LDA met shrinkage=0.1
Fold 3 | Shrinkage 0.1 | KL-score: 1.15955
Training LDA met shrinkage=0.3
Fold 3 | Shrinkage 0.3 | KL-score: 1.17836
Training LDA met shrinkage=0.5
Fold 3 | Shrinkage 0.5 | KL-sco